# Dinomaly: Porting a DINOv2 Feature-Reconstruction Model to Anomalib v1.x

This notebook has two parts:

1. **How this was implemented** — a step-by-step account of how the *Dinomaly* model was
   ported from anomalib's v2.x `main` branch (where it originally shipped) down to this
   `feature/dinomaly-v1` branch, which targets **anomalib < 2** (the pre-v2,
   `AnomalyModule`-based API). Useful if you need to port another model the same way, or
   just want to understand why the code in `src/anomalib/models/image/dinomaly/` looks the
   way it does.
2. **How to use it** — a standard train / test / infer walkthrough, in the same style as
   the other notebooks under `notebooks/200_models/`.

> Dinomaly ([paper](https://arxiv.org/abs/2405.14325),
> [original code](https://github.com/guojiajeremy/Dinomaly)) is a Vision-Transformer
> feature-reconstruction model. A frozen, pretrained **DINOv2** ViT encodes normal images;
> a small trainable bottleneck + ViT decoder is trained to reconstruct the encoder's
> mid-level features. At inference, regions where the decoder fails to reconstruct the
> encoder's features (measured by cosine similarity) are flagged as anomalous.

A reusable, generalized version of everything below is also written up as a Claude Code
skill at
[`.claude/skills/port-v2-model-to-anomalib-v1/SKILL.md`](../../.claude/skills/port-v2-model-to-anomalib-v1/SKILL.md).

## Part 1 — Implementation journey

### Why a port was needed

anomalib's `main` branch is now v2.x, which uses a redesigned Lightning API
(`AnomalibModule`, dataclass batches, `PreProcessor`/`PostProcessor`/`Evaluator`/
`Visualizer`). Dinomaly was added there first. This repo's `feature/dinomaly-v1` branch was
created from the `v1.2.0` tag — anomalib **< 2**, using the older `AnomalyModule` API — so
that anomalib<2 users get Dinomaly too. The goal was to reproduce the model faithfully
while adapting only what the older API actually requires.

### Step 0 — Sandbox environment fix (git-lfs)

Before anything else, plain `git status` in this sandbox failed with
`git-lfs filter-process: 1: git-lfs: not found` — the repo's global git config points at a
`git-lfs` binary that isn't installed, and there was no `sudo` to install it. Rather than
get stuck, the local/global LFS filter was relaxed (safe here since no LFS-tracked binary
assets were touched):

```bash
git config --global --unset filter.lfs.process
git config --global filter.lfs.required false
git config --global filter.lfs.smudge cat
git config --global filter.lfs.clean cat
```

### Step 1 — Found the target branch

`feature/dinomaly-v1` already existed locally, created from the `v1.2.0` tag
(confirmed via `git reflog show feature/dinomaly-v1` → `branch: Created from v1.2.0`), but
contained no Dinomaly code yet — it was the intended landing branch for this work. A `git
worktree` was used to have both `main` (source of the v2 implementation) and this branch
checked out simultaneously:

```bash
git worktree add /path/to/scratch/anomalib-v1 feature/dinomaly-v1
```

### Step 2 — Read and classified every v2 source file

Every file under `src/anomalib/models/image/dinomaly/` on `main` was read and put into one
of three buckets:

| Bucket | Files | Action |
|---|---|---|
| Pure PyTorch, no anomalib API dependency | `components/layers.py`, `loss.py`, `optimizer.py`, `vision_transformer.py` | Copied **verbatim** |
| Uses old, stable anomalib utilities (`GaussianBlur2d`, `DownloadInfo`, `DownloadProgressBar`) | `torch_model.py`, `components/dinov2_loader.py` | Copied with only the v2-only bits removed (see Step 4) |
| Lightning-module glue tied to the new API | `lightning_model.py` | Rewritten against the v1 `AnomalyModule` base (see Step 5) |

This 80/20 split is what made the port tractable: the actual DINOv2 architecture,
bottleneck, decoder, cosine-hard-mining loss, custom `StableAdamW` optimizer and
`WarmCosineScheduler` are 100% framework-agnostic and needed **zero** changes.

### Step 3 — Studied the v1.x `AnomalyModule` API

Read `src/anomalib/models/components/base/anomaly_module.py` and a structurally similar
existing v1 model (`reverse_distillation`, another encoder/bottleneck/decoder
reconstruction model) to learn the target conventions: how models are built
(`__init__` vs. deferred `_setup()`), how batches flow (plain `dict`, not a dataclass), how
transforms are configured, and how registration works (`AnomalyModule.__subclasses__()` —
no separate registry).

### Step 4 — Ported `torch_model.py` and `components/`

`components/layers.py`, `loss.py`, `optimizer.py`, `vision_transformer.py` and
`components/__init__.py` were copied unchanged. `components/dinov2_loader.py` was also
copied unchanged — its only anomalib dependencies
(`anomalib.data.utils.DownloadInfo`, `anomalib.data.utils.download.DownloadProgressBar`)
already exist identically in v1.2.0.

`torch_model.py` needed exactly one change: the v2 version returns
`anomalib.data.InferenceBatch(pred_score=..., anomaly_map=...)` at inference time — a
dataclass that doesn't exist pre-v2. It now returns a plain
`tuple[pred_score, anomaly_map]` instead; everything else (the DINOv2 encoder forward
pass, feature fusion, bottleneck, decoder, cosine-similarity anomaly map, Gaussian
smoothing, top-1%-mean score aggregation) is untouched.

### Step 5 — Adapted `lightning_model.py`

This is where the actual API surface differs. Key changes, side by side:

| v2 (`main`) | v1.x (this branch) |
|---|---|
| `class Dinomaly(AnomalibModule)` | `class Dinomaly(AnomalyModule)` |
| `__init__(..., pre_processor=True, post_processor=True, evaluator=True, visualizer=True)` | `__init__(...)` — those four concepts don't exist pre-v2, dropped entirely |
| `batch.image`, `batch.update(pred_score=..., anomaly_map=...)` | `batch["image"]`, `batch["pred_scores"] = ...` / `batch["anomaly_maps"] = ...` (**plural** keys) |
| `@classmethod configure_pre_processor(cls, image_size) -> PreProcessor` | `@staticmethod configure_transforms(image_size) -> Transform` (matches the convention used by every other v1 model that overrides it, and keeps `ruff`'s `PLR6301` happy since the method never touches `self`) |
| Score/threshold/metric wiring via `Evaluator`/`PostProcessor`/`Visualizer` objects | Handled automatically by Engine callbacks in v1 (`callbacks/post_processor.py`, `thresholding.py`, `metrics.py`) — nothing to configure in the module. One subtlety: the default `pred_scores` derived from `anomaly_maps` is a plain per-image **max**; Dinomaly's own top-1%-mean aggregation is computed explicitly in `validation_step` instead of relying on that default. |
| `self.log(...)`, `configure_optimizers()`, `self.trainer.max_epochs/max_steps`, `learning_type`, `trainer_arguments` | Unchanged — plain Lightning `LightningModule` API, identical across versions |

The training hyperparameters (`StableAdamW`, `WarmCosineScheduler`, gradient clipping,
freezing everything except the bottleneck + decoder) are untouched from the v2 version.

### Step 6 — Registration

Two files, both requiring alphabetical-order inserts:

- `src/anomalib/models/image/__init__.py` — `from .dinomaly import Dinomaly` + `__all__`
- `src/anomalib/models/__init__.py` — same, one level up

There's no separate model registry: `get_available_models()` / `get_model("dinomaly")`
both work by walking `AnomalyModule.__subclasses__()`, so importing the class **is** the
registration. This also means `dinomaly` is automatically picked up by
`tests/integration/model/test_models.py`, which fits/tests/predicts/exports every
registered model.

### Step 7 — Config, docs, README

- `configs/model/dinomaly.yaml` — the jsonargparse CLI config (`model.class_path`,
  `model.init_args`, `metrics.pixel: [AUROC]`, and `trainer` overrides for `max_steps`,
  `gradient_clip_val`, `val_check_interval`).
- `docs/source/markdown/guides/reference/models/image/dinomaly.md` — an `automodule` stub,
  registered as a card + toctree entry in that directory's `index.md`.
- The model's `README.md` was copied from `main` with one fix: the CLI usage line used
  `--data MVTecAD`, but the v1.x datamodule class is named `MVTec`, and v1 configs don't
  default to Dinomaly's step-based training, so it now reads:
  `anomalib train --model Dinomaly --data MVTec --data.category <category> --trainer.max_steps 5000`.

### Step 8 — A real dependency bug, found empirically

`pyproject.toml` left `timm` **unconstrained**. Testing surfaced
`TypeError: Attention.__init__() got an unexpected keyword argument 'proj_bias'` against
an older installed `timm` — Dinomaly's decoder blocks pass `proj_bias` into timm's
`Attention`, an argument that doesn't exist in every timm release. Rather than guess a
version, a quick loop installed several candidate versions and checked the signature
directly:

```python
import inspect
from timm.models.vision_transformer import Attention
print("proj_bias" in inspect.signature(Attention.__init__).parameters)
```

`timm==1.0.12` → `False`, `timm==1.0.13` → `True`. `pyproject.toml` now pins
`timm>=1.0.13` with a comment explaining why.

### Step 9 — Testing, and the environment traps along the way

The sandbox's default Python environment has anomalib **v2.1.0** installed — importing
plain `anomalib` there resolves the wrong major version entirely, so testing had to
`PYTHONPATH`-shadow the v1 worktree's `src/` ahead of any installed copy.

A few dead ends before landing on a working setup:

- A fresh `uv venv --system-site-packages` + `uv pip install timm` **without** `--no-deps`
  silently pulled in a newer torch/torchvision/CUDA stack, which broke a prebuilt
  `flash_attn` `.so` that `anomalib.metrics` transitively imports via `kornia` — a
  completely unrelated import chain several layers from anything actually touched by the
  port. Fix: `uv pip install --no-deps timm` to upgrade *only* the one outdated package.
- Even then, a leftover `numpy 2.x` vs. a `matplotlib` binary compiled against `numpy 1.x`
  caused an unrelated `ImportError`. Fix: pin `numpy<2` in that venv.
- The cleanest fix, in the end, was noticing an existing `conda` environment
  (`anomalib_dataset`) that already had a working anomalib 1.1.0 install with no broken
  `flash_attn` binary at all — `PYTHONPATH`-shadowing the worktree's `src/` into *that* env
  avoided the whole rabbit hole.

Once the environment was sane, the actual test sequence was:

1. **Architecture / forward-pass smoke test**, with the DINOv2 pretrained-weight
   *download* mocked out (`unittest.mock.patch.object(DinoV2Loader, "_load_weights",
   noop)`) so it doesn't depend on network access — build `DinomalyModel`, run a training
   forward + `.backward()`, then an eval forward, and check output shapes.
2. **One real, unmocked `get_model("dinomaly")` call** — this actually downloaded real
   DINOv2-small-reg weights (~88 MB) and loaded them into the architecture, catching any
   shape/naming mismatch the mocked test couldn't.
3. **A real 2-step `lightning.pytorch.Trainer.fit()`** against a tiny dummy `Dataset`
   yielding `{"image": torch.randn(...)}` batches — this is what actually exercises
   `configure_optimizers`, `training_step`, `self.log`, and gradient updates; a hand-rolled
   fake `Trainer` object can't fake `self.log`'s internals (`trainer.barebones`, the logger
   connector, etc.) convincingly, so a real (tiny, GPU, 2-step) `Trainer` was used instead.
4. `validation_step` was checked for correct `pred_scores`/`anomaly_maps` keys and shapes.
5. `"dinomaly" in get_available_models()` confirmed registration.

All of the above passed.

### Step 10 — Lint

`uvx ruff check` with the *latest* ruff reported ~30 issues — almost all from preview
rules the project doesn't actually enable. The repo pins an exact version in
`.pre-commit-config.yaml` (`ruff-pre-commit rev: v0.6.2`); re-running with
`uvx ruff@0.6.2 check` cut that down to a single real finding — `PLR6301` on
`configure_transforms` for not using `self` — fixed by making it a `@staticmethod`
(matching the convention every other v1 model already follows).

### Step 11 — Wrote a reusable skill

Everything above was generalized into
[`.claude/skills/port-v2-model-to-anomalib-v1/SKILL.md`](../../.claude/skills/port-v2-model-to-anomalib-v1/SKILL.md)
so the next model port (or another contributor) doesn't have to rediscover the API diff
table, the registration mechanism, the dependency-pinning trap, or the environment
gotchas from scratch.

### Step 12 — Committed and switched the branch

Everything was staged and committed to `feature/dinomaly-v1` locally (not pushed), and the
main working-copy checkout was switched onto that branch — which is what you're looking at
now.

## Part 2 — Try it yourself

The rest of this notebook is a standard train / test / infer walkthrough, in the same
style as `notebooks/200_models/201_fastflow.ipynb`, but using the ported `Dinomaly` model.

### Installing Anomalib

The easiest way to install anomalib is via pip:

```bash
%pip install anomalib
```

If you're running this notebook directly out of this branch instead (to use the ported
code before it's released), install it in editable mode from the repo root instead:

```bash
%pip install -e .
```

In [ ]:
from pathlib import Path

# NOTE: Provide the path to the dataset root directory.
#   If the dataset is not downloaded, it will be downloaded to this directory.
dataset_root = Path.cwd().parent / "datasets" / "MVTec"

## Imports

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint
from matplotlib import pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader

from anomalib.data import MVTec, PredictDataset
from anomalib.engine import Engine
from anomalib.models import Dinomaly
from anomalib.utils.post_processing import superimpose_anomaly_map

## Data Module

Dinomaly expects a comparatively large input (448x448, center-cropped to 392x392 — see
`Dinomaly.configure_transforms`), so we let the model's own default transform handle
resizing rather than fixing `image_size` on the datamodule.

In [ ]:
datamodule = MVTec(
    root=dataset_root,
    category="bottle",
    train_batch_size=16,
    eval_batch_size=16,
    num_workers=8,
)

## Dinomaly Model

Only the bottleneck MLP and ViT decoder are trained — the DINOv2 encoder stays frozen, so
training is comparatively cheap despite the model being a Vision Transformer. Let's start
by looking at the docstring.

In [ ]:
Dinomaly??

In [ ]:
model = Dinomaly(
    encoder_name="dinov2reg_vit_base_14",
    bottleneck_dropout=0.2,
    decoder_depth=8,
)

## Callbacks

Same pattern as other models: checkpoint on the best `pixel_AUROC`.

In [ ]:
callbacks = [
    ModelCheckpoint(
        mode="max",
        monitor="pixel_AUROC",
    ),
]

## Training

Dinomaly trains for a fixed number of **steps** rather than epochs (`max_steps=5000` by
default, matching the paper) — `model.trainer_arguments` supplies the recommended
`gradient_clip_val` and disables the sanity-check validation pass, but intentionally
leaves `max_steps` for you (or the `Engine`) to set explicitly.

In [ ]:
engine = Engine(
    callbacks=callbacks,
    max_steps=5000,
    accelerator="auto",
    devices=1,
    logger=False,
    **model.trainer_arguments,
)

In [ ]:
engine.fit(datamodule=datamodule, model=model)

## Testing

Now let's check the overall performance on the test set.

In [ ]:
engine.test(datamodule=datamodule, model=model)

## Inference

Same as any other anomalib model — `PredictDataset` plus `engine.predict`.

In [ ]:
inference_dataset = PredictDataset(path=dataset_root / "bottle/test/broken_large/000.png")
inference_dataloader = DataLoader(dataset=inference_dataset)

In [ ]:
predictions = engine.predict(model=model, dataloaders=inference_dataloader)[0]

In [ ]:
print(
    f'Image Shape: {predictions["image"].shape},\n'
    f'Anomaly Map Shape: {predictions["anomaly_maps"].shape}, \n'
    f'Predicted Mask Shape: {predictions["pred_masks"].shape}',
)

## Visualization

In [ ]:
image_path = predictions["image_path"][0]
image_size = predictions["image"].shape[-2:]
image = Image.open(image_path).resize(image_size)
plt.imshow(image)

In [ ]:
anomaly_map = predictions["anomaly_maps"][0]
anomaly_map = anomaly_map.cpu().numpy().squeeze()
plt.imshow(anomaly_map)

In [ ]:
heat_map = superimpose_anomaly_map(anomaly_map=anomaly_map, image=image, normalize=True)
plt.imshow(heat_map)

In [ ]:
pred_score = predictions["pred_scores"][0]
pred_label = predictions["pred_labels"][0]
print(pred_score, pred_label)

In [ ]:
pred_mask = predictions["pred_masks"][0].squeeze().cpu().numpy()
plt.imshow(pred_mask)

That's it — this notebook covered both how the Dinomaly model was ported to anomalib<2,
and how to train, test and run inference with it once ported.